## Exploratory Testing (Cyclic ID vs Standard ID)

In [34]:
# exploratory comparison - Cyclic ID vs Standard ID

# ==================================================
# Compare the cyclic ID (on a cyclic graph) vs standard ID (on an acyclic graph)
# - 20 single-gene perturbations
# - 20 multi-gene perturbations
# Total - 40 comparison runs
# ====================================================

# imports needed
import pandas as pd
import time
import matplotlib.pyplot as plt
from y0.dsl import Variable, P
import networkx as nx
from y0.algorithm.identify import Unidentifiable, identify_outcomes
from y0.algorithm.ioscm.utils import scc_to_bidirected
from y0.algorithm.identify import cyclic_id
from y0.algorithm.ioscm.utils import get_strongly_connected_components, get_graph_consolidated_districts, get_consolidated_district
from y0.graph import NxMixedGraph
from y0.algorithm.identify.utils import Identification



In [28]:
## Load E. coli GRN 

ecoli_graph = nx.read_graphml("ecoli_full_network_no_small_rna.graphml")

print(f"✓ Network loaded (NetworkX):")
print(f"  - Nodes: {ecoli_graph.number_of_nodes():,}")
print(f"  - Edges: {ecoli_graph.number_of_edges():,}")
print(f"  - Is DAG: {nx.is_directed_acyclic_graph(ecoli_graph)}")

# Convert to NxMixedGraph for causal inference
ecoli_graph = NxMixedGraph.from_edges(
    directed=list(ecoli_graph.edges())
)

print(f"\n✓ Converted to NxMixedGraph")
print(f"  - Total nodes: {len(ecoli_graph):,}")
print(f"  - Directed edges: {ecoli_graph.directed.number_of_edges():,}")
print(f"  - Undirected edges: {ecoli_graph.undirected.number_of_edges():,}")



✓ Network loaded (NetworkX):
  - Nodes: 2,976
  - Edges: 9,211
  - Is DAG: False

✓ Converted to NxMixedGraph
  - Total nodes: 2,962
  - Directed edges: 9,211
  - Undirected edges: 0


In [29]:
# pre processing step 

from y0.algorithm.ioscm.utils import scc_to_bidirected
start = time.time()

ecoli_acyclic = scc_to_bidirected(ecoli_graph)

admg_conversion_time = time.time() - start

print(f"Converted to ADMG:")
print(f"  - Conversion time: {admg_conversion_time:.2f} seconds")
print(f"  - Directed edges: {len(ecoli_acyclic.directed.edges()):,}")
print(f"  - Bidirected edges: {len(ecoli_acyclic.undirected.edges()):,}")
print(f"\n✓ ADMG ready for standard ID algorithm")


Converted to ADMG:
  - Conversion time: 2.99 seconds
  - Directed edges: 8,692
  - Bidirected edges: 484

✓ ADMG ready for standard ID algorithm


In [4]:
# load the test cases from CSV

# single gene perturbations
df_single = pd.read_csv("ecoli_benchmark_all_combined.csv")
print(f"✓ Loaded single-gene data: {len(df_single):,} queries")
print(f"  - Identifiable: {df_single['identifiable'].sum():,}")
print(f"  - Unidentifiable: {(~df_single['identifiable']).sum():,}")

# Multi-gene perturbations
df_multi = pd.read_csv('ecoli_5x5_1000samples_FINAL.csv')
print(f"\n✓ Loaded multi-gene data: {len(df_multi):,} queries")
print(f"  - Identifiable: {df_multi['identifiable'].sum():,}")
print(f"  - Unidentifiable: {(~df_multi['identifiable']).sum():,}")

# Select test queries
print("\n" + "="*60)
print("SELECTING TEST QUERIES")
print("="*60)

# Single-gene: 10 identifiable + 10 unidentifiable
single_identifiable = df_single[df_single['identifiable'] == True].sample(n=10, random_state=42)

# make random state and number of samples constants 

single_unidentifiable = df_single[df_single['identifiable'] == False].sample(n=10, random_state=42)

# Multi-gene: 10 identifiable + 10 unidentifiable
multi_identifiable = df_multi[df_multi['identifiable'] == True].sample(n=10, random_state=42)
multi_unidentifiable = df_multi[df_multi['identifiable'] == False].sample(n=10, random_state=42)

print(f"\n✓ Selected 40 test queries:")
print(f"  - Single-gene identifiable: {len(single_identifiable)}")
print(f"  - Single-gene unidentifiable: {len(single_unidentifiable)}")
print(f"  - Multi-gene identifiable: {len(multi_identifiable)}")
print(f"  - Multi-gene unidentifiable: {len(multi_unidentifiable)}")

✓ Loaded single-gene data: 555 queries
  - Identifiable: 63
  - Unidentifiable: 492

✓ Loaded multi-gene data: 1,000 queries
  - Identifiable: 893
  - Unidentifiable: 107

SELECTING TEST QUERIES

✓ Selected 40 test queries:
  - Single-gene identifiable: 10
  - Single-gene unidentifiable: 10
  - Multi-gene identifiable: 10
  - Multi-gene unidentifiable: 10


In [5]:


def compare_cyclic_vs_standardID(
    admg,                    # y0.NxMixedGraph (ADMG structure, acyclic)
    intervention,            # Gene name(s) - string or list
    target,                  # Gene name(s) - string or list
    cyclic_result_from_csv   # Boolean from CSV
):
    
    
    results = {
        'intervention': str(intervention),
        'target': str(target),
        'cyclic_id_result': cyclic_result_from_csv
    }
    
    # Run standard ID on the ADMG structure
    start = time.time()
    try:
        # Convert to Variable objects
        if isinstance(target, str):
            target_var = {Variable(target)}
        else:
            target_var = {Variable(g.strip()) for g in target}
            
        if isinstance(intervention, str):
            intervention_var = {Variable(intervention)}
        else:
            intervention_var = {Variable(g.strip()) for g in intervention}
        
        # Run standard ID directly - NO CONVERSION NEEDED!
        estimand = identify_outcomes(
            graph=admg,  
            outcomes=target_var,
            treatments=intervention_var
        )
        results['standard_id_result'] = True
        
    except Unidentifiable:
        results['standard_id_result'] = False
    except Exception as e:
        results['standard_id_result'] = False
        results['standard_id_error'] = str(e)[:200]
        
    results['standard_id_time'] = time.time() - start
    
    # Compare results
    results['results_match'] = (
        results['cyclic_id_result'] == results['standard_id_result']
    )
    
    return results

print("✓ Comparison function defined")

✓ Comparison function defined


In [6]:

comparison_results = []

# ============================================================
# [1/4] Single-gene identifiable (10 queries)
# ============================================================
print("\n" + "="*70)
print("[1/4] Running 10 single-gene IDENTIFIABLE queries...")
print("="*70)

for i, (idx, row) in enumerate(single_identifiable.iterrows(), 1):
    print(f"Query {i}/10: {row['intervention']} → {row['target']}", end="...")
    
    result = compare_cyclic_vs_standardID(
        admg=ecoli_acyclic,  # ← y0.NxMixedGraph, no conversion!
        intervention=row['intervention'],
        target=row['target'],
        cyclic_result_from_csv=row['identifiable']
    )
    
    result['query_type'] = 'single_identifiable'
    result['query_number'] = i
    comparison_results.append(result)
    
    print(f" ✓ (std: {result['standard_id_time']:.2f}s)")

# ============================================================
# [2/4] Single-gene unidentifiable (10 queries)
# ============================================================
print("\n" + "="*70)
print("[2/4] Running 10 single-gene UNIDENTIFIABLE queries...")
print("="*70)

for i, (idx, row) in enumerate(single_unidentifiable.iterrows(), 1):
    print(f"Query {i}/10: {row['intervention']} → {row['target']}", end="...")
    
    result = compare_cyclic_vs_standardID(
        admg=ecoli_acyclic,
        intervention=row['intervention'],
        target=row['target'],
        cyclic_result_from_csv=row['identifiable']
    )
    
    result['query_type'] = 'single_unidentifiable'
    result['query_number'] = i
    comparison_results.append(result)
    
    print(f" ✓ (std: {result['standard_id_time']:.2f}s)")

# ============================================================
# [3/4] Multi-gene identifiable (10 queries)
# ============================================================
print("\n" + "="*70)
print("[3/4] Running 10 multi-gene IDENTIFIABLE queries...")
print("="*70)

for i, (idx, row) in enumerate(multi_identifiable.iterrows(), 1):
    interventions = [g.strip() for g in row['interventions'].split(',')]
    outcomes = [g.strip() for g in row['outcomes'].split(',')]
    
    print(f"Query {i}/10: {len(interventions)} genes → {len(outcomes)} genes", end="...")
    
    result = compare_cyclic_vs_standardID(
        admg=ecoli_acyclic,
        intervention=interventions,
        target=outcomes,
        cyclic_result_from_csv=row['identifiable']
    )
    
    result['query_type'] = 'multi_identifiable'
    result['query_number'] = i
    comparison_results.append(result)
    
    print(f" ✓ (std: {result['standard_id_time']:.2f}s)")

# ============================================================
# [4/4] Multi-gene unidentifiable (10 queries)
# ============================================================
print("\n" + "="*70)
print("[4/4] Running 10 multi-gene UNIDENTIFIABLE queries...")
print("="*70)

for i, (idx, row) in enumerate(multi_unidentifiable.iterrows(), 1):
    interventions = [g.strip() for g in row['interventions'].split(',')]
    outcomes = [g.strip() for g in row['outcomes'].split(',')]
    
    print(f"Query {i}/10: {len(interventions)} genes → {len(outcomes)} genes", end="...")
    
    result = compare_cyclic_vs_standardID(
        admg=ecoli_acyclic,
        intervention=interventions,
        target=outcomes,
        cyclic_result_from_csv=row['identifiable']
    )
    
    result['query_type'] = 'multi_unidentifiable'
    result['query_number'] = i
    comparison_results.append(result)
    
    print(f" ✓ (std: {result['standard_id_time']:.2f}s)")

# Summary
print("\n" + "="*70)
print(f"✅ COMPLETED ALL {len(comparison_results)} COMPARISONS")
print("="*70)

single_count = len([r for r in comparison_results if 'single' in r['query_type']])
multi_count = len([r for r in comparison_results if 'multi' in r['query_type']])

print(f"\n Breakdown:")
print(f"   Single-gene queries: {single_count}")
print(f"   Multi-gene queries: {multi_count}")
print(f"   Total: {len(comparison_results)}")

total_time = sum(r['standard_id_time'] for r in comparison_results)
avg_time = total_time / len(comparison_results)

print(f"\n⏱  Timing:")
print(f"   Total Standard ID time: {total_time:.2f}s")
print(f"   Average per query: {avg_time:.3f}s")




[1/4] Running 10 single-gene IDENTIFIABLE queries...
Query 1/10: glrR → eco... ✓ (std: 0.11s)
Query 2/10: rcdA → livF... ✓ (std: 0.13s)
Query 3/10: glaR → yehW... ✓ (std: 0.21s)
Query 4/10: spf → fucK... ✓ (std: 0.17s)
Query 5/10: nsrR → rclB... ✓ (std: 0.13s)
Query 6/10: btsR → cycA... ✓ (std: 0.14s)
Query 7/10: nsrR → ynbB... ✓ (std: 0.16s)
Query 8/10: nsrR → ldtC... ✓ (std: 0.30s)
Query 9/10: btsR → stfQ... ✓ (std: 0.16s)
Query 10/10: glaR → avtA... ✓ (std: 0.16s)

[2/4] Running 10 single-gene UNIDENTIFIABLE queries...
Query 1/10: gadE → yjjY... ✓ (std: 0.14s)
Query 2/10: gadW → mtlR... ✓ (std: 0.18s)
Query 3/10: slyA → speF... ✓ (std: 0.29s)
Query 4/10: fnr → cadC... ✓ (std: 0.23s)
Query 5/10: phoP → fadA... ✓ (std: 0.15s)
Query 6/10: rpoS → pyrD... ✓ (std: 0.15s)
Query 7/10: mlrA → yghG... ✓ (std: 0.13s)
Query 8/10: fnr → yidQ... ✓ (std: 0.27s)
Query 9/10: cytR → livM... ✓ (std: 0.14s)
Query 10/10: nac → sgbE... ✓ (std: 0.13s)

[3/4] Running 10 multi-gene IDENTIFIABLE queries...


In [ ]:
# ==================================================
# STEP 6: Single-Gene Results Analysis
# ==================================================

# Filter to single-gene results
single_results_df = pd.DataFrame([r for r in comparison_results if 'single' in r['query_type']])

print("\n" + "="*70)
print("SINGLE-GENE RESULTS - SUMMARY")
print("="*70)

if len(single_results_df) == 0:
    print("\n  No results found. Please run Cell 6 first.")
else:
    # Calculate key metrics
    total = len(single_results_df)
    matches = single_results_df['results_match'].sum()
    match_rate = (matches / total) * 100
    
    # Split by query type
    identifiable_df = single_results_df[single_results_df['query_type'] == 'single_identifiable']
    unidentifiable_df = single_results_df[single_results_df['query_type'] == 'single_unidentifiable']
    
    # Overall summary
    print(f"\n Overall:")
    print(f"   Queries tested: {total}")
    print(f"   Agreement rate: {matches}/{total} ({match_rate:.1f}%)")
    print(f"   Avg time/query: {single_results_df['standard_id_time'].mean():.3f}s")
    
    # Identifiable queries
    if len(identifiable_df) > 0:
        ident_matches = identifiable_df['results_match'].sum()
        print(f"\n✅ IDENTIFIABLE (Cyclic ID = YES):")
        print(f"   Tested: {len(identifiable_df)}")
        print(f"   Standard ID agreed: {ident_matches}/{len(identifiable_df)} ({ident_matches/len(identifiable_df)*100:.1f}%)")
    
    # Unidentifiable queries
    if len(unidentifiable_df) > 0:
        unident_matches = unidentifiable_df['results_match'].sum()
        print(f"\n❌ UNIDENTIFIABLE (Cyclic ID = NO):")
        print(f"   Tested: {len(unidentifiable_df)}")
        print(f"   Standard ID agreed: {unident_matches}/{len(unidentifiable_df)} ({unident_matches/len(unidentifiable_df)*100:.1f}%)")
    
    # Key insight
    print(f"\n💡 KEY INSIGHT:")
    if match_rate == 100:
        print(f"   ✅ Perfect agreement! Standard ID validates Cyclic ID.")
    elif match_rate >= 50:
        # Check the pattern
        ident_agree_pct = (identifiable_df['results_match'].sum() / len(identifiable_df) * 100) if len(identifiable_df) > 0 else 0
        unident_agree_pct = (unidentifiable_df['results_match'].sum() / len(unidentifiable_df) * 100) if len(unidentifiable_df) > 0 else 0
    
    
    # Sample results table
    print(f"\n Sample Results Table:")
    print("-" * 70)
    display(single_results_df[['query_number', 'intervention', 'target', 
                                'cyclic_id_result', 'standard_id_result', 
                                'results_match']].head(5))
    
    # Save results
    single_results_df.to_csv('single_gene_comparison_results.csv', index=False)
    print(f"\n Saved: single_gene_comparison_results.csv")
    
    print("\n" + "="*70)


SINGLE-GENE RESULTS - SUMMARY

 Overall:
   Queries tested: 20
   Agreement rate: 10/20 (50.0%)
   Avg time/query: 0.306s

✅ IDENTIFIABLE (Cyclic ID = Agree(Yes)):
   Tested: 10
   Standard ID agreed: 10/10 (100.0%)

❌ UNIDENTIFIABLE (Cyclic ID does not Agree(No)):
   Tested: 10
   Standard ID agreed: 0/10 (0.0%)

💡 KEY INSIGHT:

 Sample Results Table:
----------------------------------------------------------------------


,query_number,intervention,target,cyclic_id_result,standard_id_result,results_match
0,1,glrR,eco,True,True,True
1,2,rcdA,livF,True,True,True
2,3,glaR,yehW,True,True,True
3,4,spf,fucK,True,True,True
4,5,nsrR,rclB,True,True,True


In [7]:
# analyzing all of the queries (40)
from datetime import datetime
import json

results_df = pd.DataFrame(comparison_results)

print("\n" + "="*70)
print("COMPLETE RESULTS - CYCLIC ID vs STANDARD ID")
print("="*70)

# Overall summary
total = len(results_df)
matches = results_df['results_match'].sum()
match_rate = (matches / total) * 100

print(f"\n📊 OVERALL (All 40 queries):")
print(f"   Queries tested: {total}")
print(f"   Agreement rate: {matches}/{total} ({match_rate:.1f}%)")
print(f"   Disagreements: {total - matches}")
print(f"   Avg time/query: {results_df['standard_id_time'].mean():.3f}s")
print(f"   Total time: {results_df['standard_id_time'].sum():.2f}s")

# ============================================================
# Breakdown by Perturbation Type (Single vs Multi)
# ============================================================
print(f"\n" + "-"*70)
print("📋 BY PERTURBATION TYPE:")
print("-"*70)

single_df = results_df[results_df['query_type'].str.contains('single')]
multi_df = results_df[results_df['query_type'].str.contains('multi')]

print(f"\n🔹 SINGLE-GENE Perturbations:")
print(f"   Queries: {len(single_df)}")
print(f"   Agreement: {single_df['results_match'].sum()}/{len(single_df)} ({single_df['results_match'].sum()/len(single_df)*100:.1f}%)")
print(f"   Avg time: {single_df['standard_id_time'].mean():.3f}s")

print(f"\n🔸 MULTI-GENE Perturbations:")
print(f"   Queries: {len(multi_df)}")
print(f"   Agreement: {multi_df['results_match'].sum()}/{len(multi_df)} ({multi_df['results_match'].sum()/len(multi_df)*100:.1f}%)")
print(f"   Avg time: {multi_df['standard_id_time'].mean():.3f}s")

# ============================================================
# Breakdown by All 4 Categories
# ============================================================
print(f"\n" + "-"*70)
print("📋 BY CATEGORY (Detailed):")
print("-"*70)

categories = [
    ('single_identifiable', '✅ Single-gene IDENTIFIABLE'),
    ('single_unidentifiable', '❌ Single-gene UNIDENTIFIABLE'),
    ('multi_identifiable', '✅ Multi-gene IDENTIFIABLE'),
    ('multi_unidentifiable', '❌ Multi-gene UNIDENTIFIABLE')
]

for cat_name, cat_label in categories:
    cat_df = results_df[results_df['query_type'] == cat_name]
    if len(cat_df) > 0:
        matches_cat = cat_df['results_match'].sum()
        total_cat = len(cat_df)
        rate = (matches_cat / total_cat * 100)
        
        print(f"\n{cat_label}:")
        print(f"   Tested: {total_cat}")
        print(f"   Standard ID agreed: {matches_cat}/{total_cat} ({rate:.1f}%)")
        print(f"   Avg time: {cat_df['standard_id_time'].mean():.3f}s")

# ============================================================
# Pattern Analysis
# ============================================================
print(f"\n" + "-"*70)
print("💡 KEY PATTERNS:")
print("-"*70)

# Calculate agreement rates for identifiable vs unidentifiable
ident_df = results_df[results_df['query_type'].str.contains('identifiable') & 
                      ~results_df['query_type'].str.contains('unidentifiable')]
unident_df = results_df[results_df['query_type'].str.contains('unidentifiable')]

ident_agree = ident_df['results_match'].sum()
ident_total = len(ident_df)
unident_agree = unident_df['results_match'].sum()
unident_total = len(unident_df)

ident_agree_rate = (ident_agree / ident_total * 100) if ident_total > 0 else 0
unident_agree_rate = (unident_agree / unident_total * 100) if unident_total > 0 else 0

print(f"\n When Cyclic ID says IDENTIFIABLE ({ident_total} queries):")
print(f"   Standard ID agreed: {ident_agree}/{ident_total} ({ident_agree_rate:.1f}%)")

print(f"\n When Cyclic ID says UNIDENTIFIABLE ({unident_total} queries):")
print(f"   Standard ID agreed: {unident_agree}/{unident_total} ({unident_agree_rate:.1f}%)")



# Check for single vs multi differences
if len(single_df) > 0 and len(multi_df) > 0:
    single_match_rate = (single_df['results_match'].sum() / len(single_df) * 100)
    multi_match_rate = (multi_df['results_match'].sum() / len(multi_df) * 100)
    
    if abs(single_match_rate - multi_match_rate) > 10:
        print(f"\n PERTURBATION TYPE DIFFERENCE:")
        print(f"   → Single-gene: {single_match_rate:.1f}% agreement")
        print(f"   → Multi-gene: {multi_match_rate:.1f}% agreement")
        if multi_match_rate > single_match_rate:
            print(f"   → Multi-gene perturbations show BETTER agreement")
        else:
            print(f"   → Single-gene perturbations show BETTER agreement")

# ============================================================
# Sample Results Table
# ============================================================
print(f"\n" + "-"*70)
print("📋 SAMPLE RESULTS (3 from each category):")
print("-"*70)

for cat_name, cat_label in categories:
    cat_df = results_df[results_df['query_type'] == cat_name]
    if len(cat_df) > 0:
        print(f"\n{cat_label}:")
        display(cat_df[['query_number', 'intervention', 'target', 
                        'cyclic_id_result', 'standard_id_result', 
                        'results_match', 'standard_id_time']].head(3))

# ============================================================
# Save Results
# ============================================================
print(f"\n" + "="*70)

# Save complete results
results_df.to_csv('complete_comparison_results.csv', index=False)
print(f"💾 Complete results saved: complete_comparison_results.csv")

# Save summary statistics
summary_stats = {
    'total_queries': int(total),
    'agreements': int(matches),
    'disagreements': int(total - matches),
    'agreement_rate_percent': float(match_rate),
    'single_gene_queries': int(len(single_df)),
    'multi_gene_queries': int(len(multi_df)),
    'single_gene_agreement_rate': float(single_df['results_match'].sum()/len(single_df)*100) if len(single_df) > 0 else 0,
    'multi_gene_agreement_rate': float(multi_df['results_match'].sum()/len(multi_df)*100) if len(multi_df) > 0 else 0,
    'identifiable_agreement_rate': float(ident_agree_rate),
    'unidentifiable_agreement_rate': float(unident_agree_rate),
    'mean_time_seconds': float(results_df['standard_id_time'].mean()),
    'total_time_seconds': float(results_df['standard_id_time'].sum()),
    'analysis_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

with open('comparison_summary.json', 'w') as f:
    json.dump(summary_stats, f, indent=2)
print(f"💾 Summary statistics saved: comparison_summary.json")

print(f"\n" + "="*70)
print("✅ Analysis Complete!")
print("="*70)

print(f"\n📌 KEY TAKEAWAY:")
print(f"   Tested {total} queries ({len(single_df)} single-gene, {len(multi_df)} multi-gene)")
print(f"   Overall agreement: {match_rate:.1f}%")



COMPLETE RESULTS - CYCLIC ID vs STANDARD ID

📊 OVERALL (All 40 queries):
   Queries tested: 40
   Agreement rate: 20/40 (50.0%)
   Disagreements: 20
   Avg time/query: 0.184s
   Total time: 7.34s

----------------------------------------------------------------------
📋 BY PERTURBATION TYPE:
----------------------------------------------------------------------

🔹 SINGLE-GENE Perturbations:
   Queries: 20
   Agreement: 10/20 (50.0%)
   Avg time: 0.173s

🔸 MULTI-GENE Perturbations:
   Queries: 20
   Agreement: 10/20 (50.0%)
   Avg time: 0.194s

----------------------------------------------------------------------
📋 BY CATEGORY (Detailed):
----------------------------------------------------------------------

✅ Single-gene IDENTIFIABLE:
   Tested: 10
   Standard ID agreed: 10/10 (100.0%)
   Avg time: 0.166s

❌ Single-gene UNIDENTIFIABLE:
   Tested: 10
   Standard ID agreed: 0/10 (0.0%)
   Avg time: 0.181s

✅ Multi-gene IDENTIFIABLE:
   Tested: 10
   Standard ID agreed: 10/10 (100.0%)
 

,query_number,intervention,target,cyclic_id_result,standard_id_result,results_match,standard_id_time
0,1,glrR,eco,True,True,True,0.107350
1,2,rcdA,livF,True,True,True,0.129667
2,3,glaR,yehW,True,True,True,0.211541



❌ Single-gene UNIDENTIFIABLE:


,query_number,intervention,target,cyclic_id_result,standard_id_result,results_match,standard_id_time
10,1,gadE,yjjY,False,True,False,0.144123
11,2,gadW,mtlR,False,True,False,0.175122
12,3,slyA,speF,False,True,False,0.288025



✅ Multi-gene IDENTIFIABLE:


,query_number,intervention,target,cyclic_id_result,standard_id_result,results_match,standard_id_time
20,1,"['ycaM', 'tufB', 'ydhV', 'hslO', 'blr']","['atoC', 'cysH', 'chpS', 'pitA', 'yqeH']",True,True,True,0.125756
21,2,"['mdtF', 'nuoK', 'pgk', 'fimB', 'lnt']","['ydeO', 'lacZ', 'pyrC', 'yihM', 'sthA']",True,True,True,0.261182
22,3,"['pabA', 'rrsC', 'livJ', 'ybeZ', 'ytfP']","['ptrA', 'iscS', 'argU', 'slyB', 'waaF']",True,True,True,0.129277



❌ Multi-gene UNIDENTIFIABLE:


,query_number,intervention,target,cyclic_id_result,standard_id_result,results_match,standard_id_time
30,1,"['patD', 'ydiL', 'pdeB', 'paaX', 'rstA']","['yagE', 'malF', 'frmR', 'thiQ', 'ubiU']",False,True,False,0.146495
31,2,"['qseB', 'ydfX', 'ybiX', 'rutR', 'rpiA']","['tauA', 'thrL', 'ileV', 'xapB', 'melB']",False,True,False,0.315457
32,3,"['lon', 'ydgI', 'cysI', 'cbrC', 'ihfA']","['ykgO', 'fadA', 'greB', 'erpA', 'narJ']",False,True,False,0.359167



💾 Complete results saved: complete_comparison_results.csv
💾 Summary statistics saved: comparison_summary.json

✅ Analysis Complete!

📌 KEY TAKEAWAY:
   Tested 40 queries (20 single-gene, 20 multi-gene)
   Overall agreement: 50.0%


## In-Depth Analysis of Unidentifiable Results
---

**Objective:** Trace through Cyclic ID and Standard ID algorithms line-by-line for queries where they disagree.

**Approach:**
- Select representative unidentifiable queries
- For each query, step through both algorithms
- At each step: Print expected vs. actual outputs

**Key Questions:**
1. Where exactly does Cyclic ID fail? (Which line?)
2. Why doesn't Standard ID fail at the same point and why does it pass?
3. Does a conversion to an acyclic graph change ancestral relationships? 

In [12]:
# get the unidentifiable queries from the results

results_df = pd.read_csv('complete_comparison_results.csv')


unidentifiable_queries_df = results_df[results_df['cyclic_id_result'] == False].copy()

print(f"\nLoaded Results:")
print(f"  - Total queries: {len(results_df)}")
print(f"  - Unidentifiable queries: {len(unidentifiable_queries_df)}")
print(f"  - Single-gene unidentifiable: {len(unidentifiable_queries_df[unidentifiable_queries_df['query_type'] == 'single_unidentifiable'])}")
print(f"  - Multi-gene unidentifiable: {len(unidentifiable_queries_df[unidentifiable_queries_df['query_type'] == 'multi_unidentifiable'])}")

# select first 3 single-gene for analysis
single_gene_unidentifiable = unidentifiable_queries_df[unidentifiable_queries_df['query_type'] == 'single_unidentifiable']
print(f"\nSelecting 3 queries to test first")
for i, (idx, row) in enumerate(single_gene_unidentifiable.head(3).iterrows(), 1):
    print(f"  {i}. {row['intervention']} → {row['target']}")



Loaded Results:
  - Total queries: 40
  - Unidentifiable queries: 20
  - Single-gene unidentifiable: 10
  - Multi-gene unidentifiable: 10

Selecting 3 queries to test first
  1. gadE → yjjY
  2. gadW → mtlR
  3. slyA → speF


### Computing Graph Properties

Before investigating individual queries, we need to precompute structural properties of the E. coli network:

1. **Strongly Connected Components (SCCs)** - Identify feedback loops
2. **Consolidated Districts** - Find confounded regions

**Why this matters:**
- The 68-node SCC (#264) is the "master regulatory hub"
- Contains major regulators like crp, fnr, hns


In [13]:
# computing the graph properties

print("\n[1] Strongly Connected Components...")
sccs = get_strongly_connected_components(ecoli_graph)
print(f"    Total SCCs: {len(sccs)}")

# count SCCs with feedback (size > 1)
large_sccs = [scc for scc in sccs if len(scc) > 1]
print(f"    SCCs with feedback (size > 1): {len(large_sccs)}")

# Find largest SCC
largest_scc_size = max(len(scc) for scc in sccs)
largest_scc_idx = [i for i, scc in enumerate(sccs) if len(scc) == largest_scc_size][0]
print(f"    Largest SCC: SCC #{largest_scc_idx} with {largest_scc_size} nodes")


# consolidated districts
print("\n[2] Consolidated Districts...")
all_districts = get_graph_consolidated_districts(ecoli_graph)
print(f"    Total districts: {len(all_districts)}")


[1] Strongly Connected Components...
    Total SCCs: 2881
    SCCs with feedback (size > 1): 14
    Largest SCC: SCC #2376 with 68 nodes

[2] Consolidated Districts...
    Total districts: 2881


In [15]:
def investigate_query_with_actual_implementation(intervention, target, cyclic_graph, admg_graph, sccs):
    """
    Trace through BOTH Cyclic ID and Standard ID algorithms for comparison.
    Handles both single-gene and multi-gene queries.
    """
    
    # Handle single or multi-gene inputs
    if isinstance(intervention, str):
        interventions = [intervention]
    else:
        interventions = intervention
    
    if isinstance(target, str):
        targets = [target]
    else:
        targets = target
    
    # Format for display
    interv_display = interventions[0] if len(interventions) == 1 else f"{interventions[0]},... ({len(interventions)} genes)"
    target_display = targets[0] if len(targets) == 1 else f"{targets[0]},... ({len(targets)} genes)"
    
    print("\n" + "="*70)
    print(f"ALGORITHM COMPARISON: {interv_display} → {target_display}")
    print("="*70)
    
    # Convert to Variable objects
    interv_vars = {Variable(i) for i in interventions}
    target_vars = {Variable(t) for t in targets}
    
    # =====================================================================
    # PART 1: CYCLIC ID ALGORITHM
    # =====================================================================
    print("\n" + "="*70)
    print("PART 1: CYCLIC ID (on cyclic graph)")
    print("="*70)
    
    if len(interventions) == 1 and len(targets) == 1:
        print(f"\nQuery: P({targets[0]} | do({interventions[0]}))")
    else:
        print(f"\nQuery: P({{{', '.join(targets)}}} | do({{{', '.join(interventions)}}}))")
    
    # ---------------------------------------------------------------------
    # Line 2: Validate preconditions
    # ---------------------------------------------------------------------
    print("\n[Line 2] Validate Preconditions")
    print("-"*70)
    
    all_nodes = set(cyclic_graph.nodes())
    
    outcomes_in_graph = target_vars.issubset(all_nodes)
    interventions_in_graph = interv_vars.issubset(all_nodes)
    disjoint = not (target_vars & interv_vars)
    
    print("\nExpected:")
    print("  Y ⊆ V: True (outcomes must be in graph)")
    print("  W ⊆ V: True (interventions must be in graph)")
    print("  Y ∩ W = ∅: True (outcomes and interventions must be disjoint)")
    
    print("\nActual:")
    print(f"  Y ⊆ V: {outcomes_in_graph} {'✓' if outcomes_in_graph else '✗'}")
    print(f"  W ⊆ V: {interventions_in_graph} {'✓' if interventions_in_graph else '✗'}")
    print(f"  Y ∩ W = ∅: {disjoint} {'✓' if disjoint else '✗'}")
    
    # ---------------------------------------------------------------------
    # Line 3: Ancestral closure
    # ---------------------------------------------------------------------
    print("\n[Line 3] Compute H in G\\W")
    print("-"*70)
    
    graph_minus_interventions = cyclic_graph.remove_nodes_from(interv_vars)
    ancestral_closure = graph_minus_interventions.ancestors_inclusive(target_vars)
    
    any_interv_in_h = any(v in ancestral_closure for v in interv_vars)
    
    print("\nExpected:")
    print("  H = An_G\\W(Y) should include all ancestors of target(s)")
    print("  W ∉ H: True (interventions removed before computing ancestors)")
    print("  If targets downstream of 68-node hub → H will be large (50-90 nodes)")
    
    print("\nActual:")
    print(f"  |H| = {len(ancestral_closure)} nodes")
    print(f"  W ∈ H: {any_interv_in_h} {'✗ ERROR' if any_interv_in_h else '✓'}")
    
    # ---------------------------------------------------------------------
    # Line 4: Districts
    # ---------------------------------------------------------------------
    print("\n[Line 4] Get Districts of H")
    print("-"*70)
    
    h_subgraph = graph_minus_interventions.subgraph(ancestral_closure)
    consolidated_districts = get_graph_consolidated_districts(h_subgraph)
    
    district_sizes = sorted([len(d) for d in consolidated_districts], reverse=True)
    
    print("\nExpected:")
    print("  H decomposes into consolidated districts")
    print("  If hub genes in H → expect one large district (~50-68 nodes)")
    print("  Other districts typically singletons or small (1-3 nodes)")
    
    print("\nActual:")
    print(f"  Number of districts: {len(consolidated_districts)}")
    print(f"  District sizes: {district_sizes[:5]}")
    
    # ---------------------------------------------------------------------
    # Line 5-8: District loop
    # ---------------------------------------------------------------------
    print("\n[Line 5-8] District Identification")
    print("-"*70)
    
    print("\nExpected:")
    print("  For each district C:")
    print("    1. Get consolidated district D in full graph G")
    print("    2. Call IDCD(G, C, D, Q[D])")
    print("    3. If any district fails → entire query fails")
    print("  For hub district: Expect A=D condition (Line 19-20 failure)")
    
    print("\nActual:")
    
    district_results = []
    hub_district_found = False
    
    for i, district_c in enumerate(consolidated_districts, 1):
        consolidated_district_of_c = get_consolidated_district(cyclic_graph, district_c)
        any_intervention_in_district = any(v in consolidated_district_of_c for v in interv_vars)
        any_target_in_district = any(v in district_c for v in target_vars)
        
        district_results.append({
            'district_c_size': len(district_c),
            'district_d_size': len(consolidated_district_of_c),
            'intervention_in_d': any_intervention_in_district,
            'target_in_c': any_target_in_district,
        })
        
        # Check A=D for hub district
        if len(consolidated_district_of_c) == 68 and any_intervention_in_district:
            district_subgraph = cyclic_graph.subgraph(consolidated_district_of_c)
            ancestral_in_district = district_subgraph.ancestors_inclusive(district_c) & consolidated_district_of_c
            
            print(f"\n  District {i}: Hub district detected (D=68)")
            print(f"    C (outcomes in district): {len(district_c)} nodes")
            print(f"    A (ancestral closure in D): {len(ancestral_in_district)} nodes")
            print(f"    D (consolidated district): {len(consolidated_district_of_c)} nodes")
            
            if ancestral_in_district == consolidated_district_of_c:
                print(f"    → A = D: {len(ancestral_in_district)} = {len(consolidated_district_of_c)} ✓")
                print(f"    → Line 19-20 condition triggered: raise Unidentifiable")
                hub_district_found = True
    
    # Find SCCs for interventions and targets
    interv_scc_info = []
    target_scc_info = []
    
    for v in interv_vars:
        for scc_idx, scc in enumerate(sccs):
            if v in scc:
                interv_scc_info.append({'var': str(v), 'scc_idx': scc_idx, 'scc_size': len(scc)})
                break
    
    for v in target_vars:
        for scc_idx, scc in enumerate(sccs):
            if v in scc:
                target_scc_info.append({'var': str(v), 'scc_idx': scc_idx, 'scc_size': len(scc)})
                break
    
    # Count interventions in hub and targets that are singletons
    in_hub_count = sum(1 for info in interv_scc_info if info['scc_size'] == 68)
    singleton_count = sum(1 for info in target_scc_info if info['scc_size'] == 1)
    
    print("\n[Summary - Cyclic ID]")
    print(f"  Interventions in 68-node hub: {in_hub_count}/{len(interventions)}")
    if in_hub_count > 0:
        hub_genes = [info['var'] for info in interv_scc_info if info['scc_size'] == 68]
        print(f"    Hub genes: {', '.join(hub_genes)}")
    print(f"  Targets in singleton SCCs: {singleton_count}/{len(targets)}")
    print(f"  Predicted result: {'UNIDENTIFIABLE (A=D in hub)' if hub_district_found else 'May be identifiable'}")
    
    # =====================================================================
    # PART 2: STANDARD ID ALGORITHM
    # =====================================================================
    print("\n" + "="*70)
    print("PART 2: STANDARD ID (on ADMG from scc_to_bidirected)")
    print("="*70)
    
    if len(interventions) == 1 and len(targets) == 1:
        print(f"\nQuery: P({targets[0]} | do({interventions[0]}))")
    else:
        print(f"\nQuery: P({{{', '.join(targets)}}} | do({{{', '.join(interventions)}}}))")
    
    # ---------------------------------------------------------------------
    # Graph structure
    # ---------------------------------------------------------------------
    print("\n[Graph Structure After Conversion]")
    print("-"*70)
    
    print("\nExpected:")
    print("  scc_to_bidirected() should:")
    print("    - Remove all directed edges within SCCs")
    print("    - Add bidirected edges for confounding")
    print("    - Result: Acyclic graph (no SCCs > 1)")
    print("    - 68-node hub → 68 singletons with bidirected edges")
    
    print("\nActual:")
    print(f"  Directed edges: {admg_graph.directed.number_of_edges():,}")
    print(f"  Bidirected edges: {admg_graph.undirected.number_of_edges()}")
    print(f"  Is acyclic: {nx.is_directed_acyclic_graph(admg_graph.directed)}")
    
    # ---------------------------------------------------------------------
    # SCCs in ADMG
    # ---------------------------------------------------------------------
    print("\n[SCCs in ADMG]")
    print("-"*70)
    
    sccs_admg = get_strongly_connected_components(admg_graph)
    largest_scc_admg = max(len(scc) for scc in sccs_admg)
    all_singletons = all(len(scc)==1 for scc in sccs_admg)
    
    print("\nExpected:")
    print("  All SCCs should be singletons (size 1)")
    print("  No feedback loops remain")
    
    print("\nActual:")
    print(f"  Total SCCs: {len(sccs_admg):,}")
    print(f"  Largest SCC: {largest_scc_admg} node")
    print(f"  All singletons: {all_singletons} {'✓' if all_singletons else '✗'}")
    
    # ---------------------------------------------------------------------
    # Run Standard ID
    # ---------------------------------------------------------------------
    print("\n[Execute Standard ID]")
    print("-"*70)
    
    print("\nExpected:")
    print("  Standard ID has NO A=D check (assumes acyclic)")
    print("  For unidentifiable queries: False positive likely")
    print("  (Treats feedback as confounding)")
    
    print("\nActual:")
    
    try:
        result_standard = identify_outcomes(
            graph=admg_graph,
            treatments=interv_vars,
            outcomes=target_vars
        )
        formula_str = str(result_standard)[:100]
        print(f"  Result: IDENTIFIABLE")
        print(f"  Formula: {formula_str}...")
        standard_result = "IDENTIFIABLE"
    except Unidentifiable as e:
        print(f"  Result: UNIDENTIFIABLE")
        standard_result = "UNIDENTIFIABLE"
    
    # ---------------------------------------------------------------------
    # Comparison
    # ---------------------------------------------------------------------
    print("\n" + "="*70)
    print("COMPARISON")
    print("="*70)
    
    cyclic_result = "UNIDENTIFIABLE" if hub_district_found else "IDENTIFIABLE"
    
    print(f"\nCyclic ID: {cyclic_result}")
    print(f"Standard ID (on scc_to_bidirected ADMG): {standard_result}")
    
    if cyclic_result != standard_result:
        print(f"\n→ DISAGREEMENT DETECTED")
        if cyclic_result == "UNIDENTIFIABLE" and standard_result == "IDENTIFIABLE":
            print(f"  Type: Standard ID FALSE POSITIVE")
            print(f"  Reason: scc_to_bidirected() removed hub structure")
            print(f"          Converted feedback (⇄) → confounding (↔)")
            print(f"          Standard ID proceeds without A=D check")
    else:
        print(f"\n→ AGREEMENT")
    
    return {
        'intervention': interv_display,
        'target': target_display,
        'num_interventions': len(interventions),
        'num_targets': len(targets),
        'interventions_in_hub': in_hub_count,
        'targets_singleton': singleton_count,
        'ancestral_closure_size': len(ancestral_closure),
        'num_districts': len(consolidated_districts),
        'cyclic_result': cyclic_result,
        'standard_result': standard_result,
        'district_results': district_results,
    }

print("Function created")

Function created


### Running trace investigation on 3 Example Queries

Now we'll trace through the algorithm for 3 representative unidentifiable queries.

**What we're looking for:**
- Which district contains the intervention gene?
- Does that district have D=68 nodes (the hub)?
- Does the A=D condition trigger in that district?
- Is the intervention gene in SCC #264 (the 68-node hub)?

In [16]:
# running on 3 example queries to test function

print("\n" + "="*70)
print("DETAILED ALGORITHM TRACES - 3 EXAMPLES")
print("="*70)

investigation_results_examples = []

for i, (idx, row) in enumerate(single_gene_unidentifiable.head(3).iterrows(), 1):
    print(f"\n\n{'='*70}")
    print(f"EXAMPLE {i} of 3")
    print('='*70)
    
    result = investigate_query_with_actual_implementation(
        intervention=row['intervention'],
        target=row['target'],
        cyclic_graph=ecoli_graph,
        admg_graph=ecoli_acyclic,
        sccs=sccs
    )
    
    investigation_results_examples.append(result)

print("\n" + "="*70)
print("✓ 3 example investigations complete")
print("="*70)


DETAILED ALGORITHM TRACES - 3 EXAMPLES


EXAMPLE 1 of 3

ALGORITHM COMPARISON: gadE → yjjY

PART 1: CYCLIC ID (on cyclic graph)

Query: P(yjjY | do(gadE))

[Line 2] Validate Preconditions
----------------------------------------------------------------------

Expected:
  Y ⊆ V: True (outcomes must be in graph)
  W ⊆ V: True (interventions must be in graph)
  Y ∩ W = ∅: True (outcomes and interventions must be disjoint)

Actual:
  Y ⊆ V: True ✓
  W ⊆ V: True ✓
  Y ∩ W = ∅: True ✓

[Line 3] Compute H in G\W
----------------------------------------------------------------------

Expected:
  H = An_G\W(Y) should include all ancestors of target(s)
  W ∉ H: True (interventions removed before computing ancestors)
  If targets downstream of 68-node hub → H will be large (50-90 nodes)

Actual:
  |H| = 56 nodes
  W ∈ H: False ✓

[Line 4] Get Districts of H
----------------------------------------------------------------------

Expected:
  H decomposes into consolidated districts
  If hub genes 

### Summary of 3 Example Investigations


**Pattern Observed:**
- All interventions are in SCC #264 (68-node master regulatory hub)
- All targets are in singleton SCCs (no feedback)
- Ancestral closure H ranges from 56-76 nodes
- H decomposes into 8-10 districts
- One district is the "hub district" with D=68 nodes
- In that hub district: A=D=68, triggering the failure when IDCD is called
- Standard ID succeeds on ADMG because all SCCs are singletons (no A=D check)

**Key Takeaway:**
The 68-node hub acts as a "black box" where individual causal effects cannot be separated from feedback. Cyclic ID correctly identifies this and it seems the standard ID algorithm is ignoring it. 

In [17]:

print("\n" + "="*70)
print("SUMMARY TABLE - 3 EXAMPLES")
print("="*70)

# Create summary dataframe
summary_3_examples = pd.DataFrame(investigation_results_examples)

# Add computed columns
summary_3_examples['hub_district'] = summary_3_examples['district_results'].apply(
    lambda dists: next((f"Outcomes={d['district_c_size']}, District Size={d['district_d_size']}" 
                       for d in dists if d['district_d_size']==68), "Not found")
)

# Display table
display_df = summary_3_examples[['intervention', 'target', 'ancestral_closure_size', 
                                  'num_districts', 'hub_district', 
                                  'cyclic_result', 'standard_result']].copy()

display_df.columns = ['Intervention', 'Target', 'Ancestral Set Size', 'Districts', 
                       'Hub District', 'Cyclic ID', 'Standard ID']

print("\n" + display_df.to_string(index=False))

print("\n" + "="*70)
print("PATTERN CONFIRMATION")
print("="*70)

print(f"\n✓ All 3 interventions in 68-node hub (SCC #264)")
print(f"✓ All 3 targets in singleton SCCs")
print(f"✓ All 3 have hub district with D=68, A=D")
print(f"✓ Cyclic ID: 3/3 UNIDENTIFIABLE (correct)")
print(f"✓ Standard ID: 3/3 IDENTIFIABLE (false positive)")
print(f"✓ Disagreement: 3/3 (100%)")


SUMMARY TABLE - 3 EXAMPLES

Intervention Target  Ancestral Set Size  Districts                  Hub District      Cyclic ID  Standard ID
        gadE   yjjY                  56          8 Outcomes=49, District Size=68 UNIDENTIFIABLE IDENTIFIABLE
        gadW   mtlR                  73         10 Outcomes=64, District Size=68 UNIDENTIFIABLE IDENTIFIABLE
        slyA   speF                  76         10 Outcomes=67, District Size=68 UNIDENTIFIABLE IDENTIFIABLE

PATTERN CONFIRMATION

✓ All 3 interventions in 68-node hub (SCC #264)
✓ All 3 targets in singleton SCCs
✓ All 3 have hub district with D=68, A=D
✓ Cyclic ID: 3/3 UNIDENTIFIABLE (correct)
✓ Standard ID: 3/3 IDENTIFIABLE (false positive)
✓ Disagreement: 3/3 (100%)


### Verify Pattern: Running 10 Single-Gene Unidentifiable Queries

To further validate these initial findings we can now run **all 10 single-gene unidentifiable queries** to confirm the pattern holds across every case.


In [18]:


print("\n" + "="*70)
print("Running 10 Single Gene perturbations Unidentifiable Queries")
print("="*70)

single_gene_results = []

for i, (idx, row) in enumerate(single_gene_unidentifiable.iterrows(), 1):
    print(f"\n{'='*70}")
    print(f"QUERY {i} of 10")
    print('='*70)
    
    result = investigate_query_with_actual_implementation(
        intervention=row['intervention'],
        target=row['target'],
        cyclic_graph=ecoli_graph,
        admg_graph=ecoli_acyclic,
        sccs=sccs
    )
    
    single_gene_results.append(result)

print("\n" + "="*70)
print("✓ All 10 single-gene investigations complete")
print("="*70)

# Create summary table
print("\n" + "="*70)
print("SUMMARY TABLE - 10 SINGLE-GENE QUERIES")
print("="*70)

summary_single = pd.DataFrame(single_gene_results)

summary_single['hub_info'] = summary_single['district_results'].apply(
    lambda dists: next((f"Outcomes={d['district_c_size']}, District Size={d['district_d_size']}" 
                       for d in dists if d['district_d_size']==68), "Hub not found")
)

summary_single['hub_genes'] = summary_single.apply(
    lambda row: f"{row['interventions_in_hub']}/{row['num_interventions']}", axis=1
)

summary_single['singleton_targets'] = summary_single.apply(
    lambda row: f"{row['targets_singleton']}/{row['num_targets']}", axis=1
)

display_single = summary_single[['intervention', 'target', 'hub_genes', 'singleton_targets',
                                  'ancestral_closure_size', 'num_districts', 'hub_info', 
                                  'cyclic_result', 'standard_result']].copy()

display_single.columns = ['Intervention', 'Target', 'In Hub', 'Singletons',
                          'H Size', 'Districts', 'Hub District', 'Cyclic ID', 'Standard ID']

print("\n" + display_single.to_string(index=False))

print("\n" + "="*70)
print("PATTERN CONFIRMATION")
print("="*70)

hub_found = (summary_single['hub_info'].str.contains('District Size=68')).sum()
cyclic_unident = (summary_single['cyclic_result'] == 'UNIDENTIFIABLE').sum()
standard_ident = (summary_single['standard_result'] == 'IDENTIFIABLE').sum()

print(f"\n✓ Hub district (D=68) found: {hub_found}/10 ({hub_found/10*100:.0f}%)")
print(f"✓ Cyclic ID unidentifiable: {cyclic_unident}/10 ({cyclic_unident/10*100:.0f}%)")
print(f"✓ Standard ID identifiable: {standard_ident}/10 ({standard_ident/10*100:.0f}%)")
print(f"✓ Disagreement: {standard_ident}/10 ({standard_ident/10*100:.0f}%)")

print(f"\n[Gene-Level Statistics]")
print(f"  Interventions in hub: {summary_single['interventions_in_hub'].sum()}/10 ({summary_single['interventions_in_hub'].sum()/10*100:.0f}%)")
print(f"  Targets in singleton SCCs: {summary_single['targets_singleton'].sum()}/10 ({summary_single['targets_singleton'].sum()/10*100:.0f}%)")

print(f"\n💡 Key Insight:")
print(f"   All single-gene interventions are in the 68-node hub,")
print(f"   triggering the A=D condition in every query.")


Running 10 Single Gene perturbations Unidentifiable Queries

QUERY 1 of 10

ALGORITHM COMPARISON: gadE → yjjY

PART 1: CYCLIC ID (on cyclic graph)

Query: P(yjjY | do(gadE))

[Line 2] Validate Preconditions
----------------------------------------------------------------------

Expected:
  Y ⊆ V: True (outcomes must be in graph)
  W ⊆ V: True (interventions must be in graph)
  Y ∩ W = ∅: True (outcomes and interventions must be disjoint)

Actual:
  Y ⊆ V: True ✓
  W ⊆ V: True ✓
  Y ∩ W = ∅: True ✓

[Line 3] Compute H in G\W
----------------------------------------------------------------------

Expected:
  H = An_G\W(Y) should include all ancestors of target(s)
  W ∉ H: True (interventions removed before computing ancestors)
  If targets downstream of 68-node hub → H will be large (50-90 nodes)

Actual:
  |H| = 56 nodes
  W ∈ H: False ✓

[Line 4] Get Districts of H
----------------------------------------------------------------------

Expected:
  H decomposes into consolidated distri

In [19]:

print("\n" + "="*70)
print("Running 10 Multi-Gene perturbations Unidentifiable Queries")
print("="*70)

multi_gene_unidentifiable = unidentifiable_queries_df[unidentifiable_queries_df['query_type'] == 'multi_unidentifiable']
multi_gene_results = []

import ast

for i, (idx, row) in enumerate(multi_gene_unidentifiable.iterrows(), 1):
    print(f"\n{'='*70}")
    print(f"QUERY {i} of 10")
    print('='*70)
    
    # Parse multi-gene interventions and targets
    interventions = ast.literal_eval(row['intervention']) if isinstance(row['intervention'], str) and row['intervention'].startswith('[') else [row['intervention']]
    targets = ast.literal_eval(row['target']) if isinstance(row['target'], str) and row['target'].startswith('[') else [row['target']]
    
    # Run investigation with FULL gene sets
    result = investigate_query_with_actual_implementation(
        intervention=interventions,  # Pass full list
        target=targets,               # Pass full list
        cyclic_graph=ecoli_graph,
        admg_graph=ecoli_acyclic,
        sccs=sccs
    )
    
    multi_gene_results.append(result)

print("\n" + "="*70)
print("✓ All 10 multi-gene investigations complete")
print("="*70)

# Create summary table
print("\n" + "="*70)
print("SUMMARY TABLE - 10 MULTI-GENE QUERIES")
print("="*70)

summary_multi = pd.DataFrame(multi_gene_results)

summary_multi['hub_info'] = summary_multi['district_results'].apply(
    lambda dists: next((f"Outcomes={d['district_c_size']}, District Size={d['district_d_size']}" 
                       for d in dists if d['district_d_size']==68), "Hub not found")
)

summary_multi['gene_counts'] = summary_multi.apply(
    lambda row: f"{row['num_interventions']}→{row['num_targets']}", axis=1
)

summary_multi['hub_genes'] = summary_multi.apply(
    lambda row: f"{row['interventions_in_hub']}/{row['num_interventions']}", axis=1
)

summary_multi['singleton_targets'] = summary_multi.apply(
    lambda row: f"{row['targets_singleton']}/{row['num_targets']}", axis=1
)

display_multi = summary_multi[['gene_counts', 'intervention', 'target', 'hub_genes', 'singleton_targets',
                                'ancestral_closure_size', 'num_districts', 'hub_info', 
                                'cyclic_result', 'standard_result']].copy()

display_multi.columns = ['Genes', 'Intervention', 'Target', 'In Hub', 'Singletons',
                          'H Size', 'Districts', 'Hub District', 'Cyclic ID', 'Standard ID']

print("\n" + display_multi.to_string(index=False))

print("\n" + "="*70)
print("PATTERN CONFIRMATION")
print("="*70)

hub_found = (summary_multi['hub_info'].str.contains('District Size=68')).sum()
cyclic_unident = (summary_multi['cyclic_result'] == 'UNIDENTIFIABLE').sum()
standard_ident = (summary_multi['standard_result'] == 'IDENTIFIABLE').sum()

# Calculate hub intervention statistics
total_interventions = summary_multi['num_interventions'].sum()
total_in_hub = summary_multi['interventions_in_hub'].sum()
total_targets = summary_multi['num_targets'].sum()
total_singletons = summary_multi['targets_singleton'].sum()

print(f"\n✓ Hub district (D=68) found: {hub_found}/10 ({hub_found/10*100:.0f}%)")
print(f"✓ Cyclic ID unidentifiable: {cyclic_unident}/10 ({cyclic_unident/10*100:.0f}%)")
print(f"✓ Standard ID identifiable: {standard_ident}/10 ({standard_ident/10*100:.0f}%)")
print(f"✓ Disagreement: {standard_ident}/10 ({standard_ident/10*100:.0f}%)")

print(f"\n[Gene-Level Statistics]")
print(f"  Total intervention genes: {total_interventions}")
print(f"  Interventions in hub: {total_in_hub}/{total_interventions} ({total_in_hub/total_interventions*100:.1f}%)")
print(f"  Total target genes: {total_targets}")
print(f"  Targets in singleton SCCs: {total_singletons}/{total_targets} ({total_singletons/total_targets*100:.1f}%)")

print(f"\nKey Insight:")
print(f"   Multi-gene queries show the SAME mechanism as single-gene:")
print(f"   Even when only {total_in_hub/total_interventions*100:.1f}% of intervention genes are in the hub,")
print(f"   the hub district triggers A=D failure for ALL queries.")


Running 10 Multi-Gene perturbations Unidentifiable Queries

QUERY 1 of 10

ALGORITHM COMPARISON: patD,... (5 genes) → yagE,... (5 genes)

PART 1: CYCLIC ID (on cyclic graph)

Query: P({yagE, malF, frmR, thiQ, ubiU} | do({patD, ydiL, pdeB, paaX, rstA}))

[Line 2] Validate Preconditions
----------------------------------------------------------------------

Expected:
  Y ⊆ V: True (outcomes must be in graph)
  W ⊆ V: True (interventions must be in graph)
  Y ∩ W = ∅: True (outcomes and interventions must be disjoint)

Actual:
  Y ⊆ V: True ✓
  W ⊆ V: True ✓
  Y ∩ W = ∅: True ✓

[Line 3] Compute H in G\W
----------------------------------------------------------------------

Expected:
  H = An_G\W(Y) should include all ancestors of target(s)
  W ∉ H: True (interventions removed before computing ancestors)
  If targets downstream of 68-node hub → H will be large (50-90 nodes)

Actual:
  |H| = 86 nodes
  W ∈ H: False ✓

[Line 4] Get Districts of H
------------------------------------------

## Testing On a Smaller Sub Graph
---


In [30]:
# creating the smaller stress response network for comparison


# Define the network
nodes = ['fur', 'fnr', 'soxR', 'oxyR', 'soxS']

edges = [
    ('oxyR', 'fur'),
    ('fur', 'soxR'),
    ('soxR', 'oxyR'),  # completes 3-gene cycle
    ('soxR', 'soxS'),
    ('fnr', 'soxR'),
    ('fnr', 'fnr'),     # self-loop
]

# Create cyclic graph
stress_graph = NxMixedGraph.from_edges(directed=edges)

print(f"\n✓ Network created:")
print(f"  - Nodes: {len(stress_graph.nodes())}")
print(f"  - Edges: {len(list(stress_graph.directed.edges()))}")
print(f"  - Is cyclic: {not nx.is_directed_acyclic_graph(stress_graph.directed)}")

# Display edge list
print(f"\nEdge list:")
for src, tgt in sorted(edges):
    note = " (self-loop)" if src == tgt else ""
    print(f"  {src} → {tgt}{note}")


✓ Network created:
  - Nodes: 5
  - Edges: 6
  - Is cyclic: True

Edge list:
  fnr → fnr (self-loop)
  fnr → soxR
  fur → soxR
  oxyR → fur
  soxR → oxyR
  soxR → soxS


In [31]:
# analyzing the graph structure

# Get SCCs
sccs = get_strongly_connected_components(stress_graph)
sccs_sorted = sorted(sccs, key=len, reverse=True)

print(f"\nStrongly Connected Components: {len(sccs)}")
print("-"*70)

for i, scc in enumerate(sccs_sorted, 1):
    scc_genes = sorted([str(v) for v in scc])
    print(f"  SCC #{i}: {scc_genes} (size={len(scc)})")
    
    if len(scc) > 1:
        print(f"    → Feedback loop detected!")
    elif len(scc) == 1 and (list(scc)[0], list(scc)[0]) in stress_graph.directed.edges():
        print(f"    → Self-loop detected!")

# Identify the 3-gene cycle
three_gene_cycle = [scc for scc in sccs if len(scc) == 3]
if three_gene_cycle:
    cycle_genes = sorted([str(v) for v in three_gene_cycle[0]])
    print(f"\n✓ 3-gene feedback loop identified: {' ↔ '.join(cycle_genes)}")
    print(f"  These genes are mutual ancestors (all are ancestors of all)")


Strongly Connected Components: 3
----------------------------------------------------------------------
  SCC #1: ['fur', 'oxyR', 'soxR'] (size=3)
    → Feedback loop detected!
  SCC #2: ['soxS'] (size=1)
  SCC #3: ['fnr'] (size=1)
    → Self-loop detected!

✓ 3-gene feedback loop identified: fur ↔ oxyR ↔ soxR
  These genes are mutual ancestors (all are ancestors of all)


In [33]:

# Convert using scc_to_bidirected
stress_acyclic = scc_to_bidirected(stress_graph)

print(f"\n✓ Conversion complete:")
print(f"  - Directed edges: {stress_acyclic.directed.number_of_edges()}")
print(f"  - Bidirected edges: {stress_acyclic.undirected.number_of_edges()}")
print(f"  - Is acyclic: {nx.is_directed_acyclic_graph(stress_acyclic.directed)}")

# Check what happened to the 3-gene cycle
print(f"\nStructural changes:")
sccs_after = get_strongly_connected_components(stress_acyclic)
print(f"  - SCCs before: {len(sccs)}")
print(f"  - SCCs after: {len(sccs_after)}")
print(f"  - Largest SCC before: {max(len(scc) for scc in sccs)}")
print(f"  - Largest SCC after: {max(len(scc) for scc in sccs_after)}")




✓ Conversion complete:
  - Directed edges: 2
  - Bidirected edges: 4
  - Is acyclic: True

Structural changes:
  - SCCs before: 3
  - SCCs after: 5
  - Largest SCC before: 3
  - Largest SCC after: 1


In [35]:

print("\n" + "="*70)
print("DEFINING TEST QUERIES")
print("="*70)

# Define the 6 test queries
test_queries = [
    {
        'query_num': 1,
        'notation': 'P(soxS | do(fnr))',
        'intervention': ['fnr'],
        'target': ['soxS'],
        'description': 'Single intervention outside cycle → target outside',
        'expected_cyclic': 'IDENTIFIABLE',
        'reasoning': 'fnr outside 3-gene cycle, no A=D'
    },
    {
        'query_num': 2,
        'notation': 'P(oxyR | do(fnr))',
        'intervention': ['fnr'],
        'target': ['oxyR'],
        'description': 'Single intervention outside cycle → target in cycle',
        'expected_cyclic': 'IDENTIFIABLE',
        'reasoning': 'fnr outside cycle, can identify effect on oxyR'
    },
    {
        'query_num': 3,
        'notation': 'P(oxyR | do(fur))',
        'intervention': ['fur'],
        'target': ['oxyR'],
        'description': 'Single intervention in cycle → target in cycle',
        'expected_cyclic': 'UNIDENTIFIABLE',
        'reasoning': 'Both in 3-gene cycle, A=D condition triggers'
    },
    {
        'query_num': 4,
        'notation': 'P(soxS | do(fnr, fur))',
        'intervention': ['fnr', 'fur'],
        'target': ['soxS'],
        'description': 'Multi-gene: one outside, one in cycle',
        'expected_cyclic': 'UNIDENTIFIABLE',
        'reasoning': 'fur in cycle creates A=D condition'
    },
    {
        'query_num': 5,
        'notation': 'P(oxyR | do(fur, soxR))',
        'intervention': ['fur', 'soxR'],
        'target': ['oxyR'],
        'description': 'Multi-gene: both in cycle',
        'expected_cyclic': 'UNIDENTIFIABLE',
        'reasoning': 'Both interventions in 3-gene cycle, A=D condition'
    },
    {
        'query_num': 6,
        'notation': 'P(oxyR, soxS | do(fnr))',
        'intervention': ['fnr'],
        'target': ['oxyR', 'soxS'],
        'description': 'Multi-target from outside cycle',
        'expected_cyclic': 'IDENTIFIABLE',
        'reasoning': 'fnr outside cycle, both targets reachable'
    }
]

print(f"\n✓ Defined {len(test_queries)} test queries:")
print("-"*70)

for q in test_queries:
    print(f"\nQuery {q['query_num']}: {q['notation']}")
    print(f"  Type: {q['description']}")
    print(f"  Expected (Cyclic ID): {q['expected_cyclic']}")
    print(f"  Reasoning: {q['reasoning']}")


DEFINING TEST QUERIES

✓ Defined 6 test queries:
----------------------------------------------------------------------

Query 1: P(soxS | do(fnr))
  Type: Single intervention outside cycle → target outside
  Expected (Cyclic ID): IDENTIFIABLE
  Reasoning: fnr outside 3-gene cycle, no A=D

Query 2: P(oxyR | do(fnr))
  Type: Single intervention outside cycle → target in cycle
  Expected (Cyclic ID): IDENTIFIABLE
  Reasoning: fnr outside cycle, can identify effect on oxyR

Query 3: P(oxyR | do(fur))
  Type: Single intervention in cycle → target in cycle
  Expected (Cyclic ID): UNIDENTIFIABLE
  Reasoning: Both in 3-gene cycle, A=D condition triggers

Query 4: P(soxS | do(fnr, fur))
  Type: Multi-gene: one outside, one in cycle
  Expected (Cyclic ID): UNIDENTIFIABLE
  Reasoning: fur in cycle creates A=D condition

Query 5: P(oxyR | do(fur, soxR))
  Type: Multi-gene: both in cycle
  Expected (Cyclic ID): UNIDENTIFIABLE
  Reasoning: Both interventions in 3-gene cycle, A=D condition

Query 6

In [36]:
# ============================================================================
# STEP 5: RUN CYCLIC ID vs STANDARD ID COMPARISON
# ============================================================================

print("\n" + "="*70)
print("RUNNING COMPARISON ON 6 TEST QUERIES")
print("="*70)

results_5gene = []

for query in test_queries:
    print(f"\n{'='*70}")
    print(f"QUERY {query['query_num']}: {query['notation']}")
    print(f"{'='*70}")
    print(f"Description: {query['description']}")
    print(f"Expected: {query['expected_cyclic']}")
    print(f"Reasoning: {query['reasoning']}")
    
    # Convert to Variable objects
    intervention_vars = {Variable(i) for i in query['intervention']}
    target_vars = {Variable(t) for t in query['target']}
    
    # -------------------------------------------------------------------------
    # Cyclic ID on cyclic graph
    # -------------------------------------------------------------------------
    print(f"\n[Cyclic ID]")
    cyclic_start = time.time()
    
    try:
        cyclic_result = cyclic_id(
            graph=stress_graph,
            outcomes=target_vars,
            interventions=intervention_vars
        )
        cyclic_identifiable = True
        cyclic_time = time.time() - cyclic_start
        print(f"  Result: IDENTIFIABLE ✓")
        print(f"  Runtime: {cyclic_time:.4f}s")
    except Unidentifiable as e:
        cyclic_identifiable = False
        cyclic_time = time.time() - cyclic_start
        error_msg = str(e)
        print(f"  Result: UNIDENTIFIABLE ✗")
        print(f"  Runtime: {cyclic_time:.4f}s")
        print(f"  Error: {error_msg[:100]}...")
        
        if "Ancestral closure equals district" in error_msg:
            print(f"  Mechanism: A=D condition detected")
    
    # -------------------------------------------------------------------------
    # Standard ID on ADMG
    # -------------------------------------------------------------------------
    print(f"\n[Standard ID on ADMG]")
    standard_start = time.time()
    
    try:
        standard_result = identify_outcomes(
            graph=stress_acyclic,
            treatments=intervention_vars,
            outcomes=target_vars
        )
        standard_identifiable = True
        standard_time = time.time() - standard_start
        print(f"  Result: IDENTIFIABLE ✓")
        print(f"  Runtime: {standard_time:.4f}s")
    except Unidentifiable as e:
        standard_identifiable = False
        standard_time = time.time() - standard_start
        print(f"  Result: UNIDENTIFIABLE ✗")
        print(f"  Runtime: {standard_time:.4f}s")
        print(f"  Error: {str(e)[:100]}...")
    
    # -------------------------------------------------------------------------
    # Compare
    # -------------------------------------------------------------------------
    agrees = cyclic_identifiable == standard_identifiable
    
    print(f"\n[Comparison]")
    print(f"  Cyclic ID: {'IDENTIFIABLE' if cyclic_identifiable else 'UNIDENTIFIABLE'}")
    print(f"  Standard ID: {'IDENTIFIABLE' if standard_identifiable else 'UNIDENTIFIABLE'}")
    
    if agrees:
        print(f"  → AGREEMENT ✓")
    else:
        print(f"  → DISAGREEMENT ⚠")
        if cyclic_identifiable == False and standard_identifiable == True:
            print(f"  → Type: Standard ID FALSE POSITIVE")
            print(f"  → Cause: scc_to_bidirected() removed 3-gene cycle structure")
    
    # Check if prediction was correct
    expected_match = (cyclic_identifiable and query['expected_cyclic'] == 'IDENTIFIABLE') or \
                     (not cyclic_identifiable and query['expected_cyclic'] == 'UNIDENTIFIABLE')
    
    if expected_match:
        print(f"  → Expected result confirmed ✓")
    else:
        print(f"  → Unexpected result! Expected {query['expected_cyclic']}")
    
    results_5gene.append({
        'query_num': query['query_num'],
        'notation': query['notation'],
        'intervention': ', '.join(query['intervention']),
        'target': ', '.join(query['target']),
        'description': query['description'],
        'expected_cyclic': query['expected_cyclic'],
        'cyclic_id': 'IDENTIFIABLE' if cyclic_identifiable else 'UNIDENTIFIABLE',
        'standard_id': 'IDENTIFIABLE' if standard_identifiable else 'UNIDENTIFIABLE',
        'agreement': agrees,
        'cyclic_time': cyclic_time,
        'standard_time': standard_time
    })

print("\n" + "="*70)
print("✓ All 6 queries tested")
print("="*70)


RUNNING COMPARISON ON 6 TEST QUERIES

QUERY 1: P(soxS | do(fnr))
Description: Single intervention outside cycle → target outside
Expected: IDENTIFIABLE
Reasoning: fnr outside 3-gene cycle, no A=D

[Cyclic ID]
  Result: IDENTIFIABLE ✓
  Runtime: 0.0139s

[Standard ID on ADMG]
  Result: IDENTIFIABLE ✓
  Runtime: 0.0022s

[Comparison]
  Cyclic ID: IDENTIFIABLE
  Standard ID: IDENTIFIABLE
  → AGREEMENT ✓
  → Expected result confirmed ✓

QUERY 2: P(oxyR | do(fnr))
Description: Single intervention outside cycle → target in cycle
Expected: IDENTIFIABLE
Reasoning: fnr outside cycle, can identify effect on oxyR

[Cyclic ID]
  Result: IDENTIFIABLE ✓
  Runtime: 0.0015s

[Standard ID on ADMG]
  Result: IDENTIFIABLE ✓
  Runtime: 0.0002s

[Comparison]
  Cyclic ID: IDENTIFIABLE
  Standard ID: IDENTIFIABLE
  → AGREEMENT ✓
  → Expected result confirmed ✓

QUERY 3: P(oxyR | do(fur))
Description: Single intervention in cycle → target in cycle
Expected: UNIDENTIFIABLE
Reasoning: Both in 3-gene cycle, A=D

In [ ]:
# results and table to summarize everything 

print("\n" + "="*70)
print("Results Summary and Final Table")
print("="*70)

df_stress_response = pd.DataFrame(results_5gene)

# Overall statistics
total = len(df_stress_response)
agreements = df_stress_response['agreement'].sum()
disagreements = total - agreements
agreement_rate = (agreements / total) * 100

print(f"\nOverall:")
print(f"  Total queries: {total}")
print(f"  Agreements: {agreements}/{total} ({agreement_rate:.1f}%)")
print(f"  Disagreements: {disagreements}/{total} ({100-agreement_rate:.1f}%)")

# Breakdown by the expected result
print(f"\nBreakdown by Cyclic ID result:")

identifiable_df = df_stress_response[df_stress_response['cyclic_id'] == 'IDENTIFIABLE']
unidentifiable_df = df_stress_response[df_stress_response['cyclic_id'] == 'UNIDENTIFIABLE']

if len(identifiable_df) > 0:
    ident_agree = identifiable_df['agreement'].sum()
    print(f"  Identifiable ({len(identifiable_df)} queries):")
    print(f"    Standard ID agrees: {ident_agree}/{len(identifiable_df)} ({ident_agree/len(identifiable_df)*100:.0f}%)")
    print(f"    Queries: {list(identifiable_df['query_num'])}")

if len(unidentifiable_df) > 0:
    unident_agree = unidentifiable_df['agreement'].sum()
    print(f"  Unidentifiable ({len(unidentifiable_df)} queries):")
    print(f"    Standard ID agrees: {unident_agree}/{len(unidentifiable_df)} ({unident_agree/len(unidentifiable_df)*100:.0f}%)")
    print(f"    Queries: {list(unidentifiable_df['query_num'])}")

# Check if predictions were correct
correct_predictions = (df_stress_response['cyclic_id'] == df_stress_response['expected_cyclic']).sum()
print(f"\nPrediction accuracy:")
print(f"  Cyclic ID matched expectations: {correct_predictions}/{total} ({correct_predictions/total*100:.0f}%)")

# Performance
print(f"\nPerformance:")
print(f"  Avg Cyclic ID time: {df_stress_response['cyclic_time'].mean():.4f}s")
print(f"  Avg Standard ID time: {df_stress_response['standard_time'].mean():.4f}s")

# Display full table
print(f"\n" + "="*70)
print("Results Table")
print("="*70)

display_df = df_stress_response[['query_num', 'notation', 'description', 
                        'expected_cyclic', 'cyclic_id', 'standard_id', 'agreement']].copy()
display_df.columns = ['Q#', 'Query', 'Description', 'Expected', 'Cyclic ID', 'Standard ID', 'Agreement']

print("\n" + display_df.to_string(index=False))

# Save results
df_stress_response.to_csv('5gene_comparison_results.csv', index=False)
print(f"\n✓ Saved: 5gene_comparison_results.csv")


RESULTS SUMMARY - 5-GENE NETWORK (6 TEST QUERIES)

Overall:
  Total queries: 6
  Agreements: 3/6 (50.0%)
  Disagreements: 3/6 (50.0%)

Breakdown by Cyclic ID result:
  Identifiable (3 queries):
    Standard ID agrees: 3/3 (100%)
    Queries: [1, 2, 6]
  Unidentifiable (3 queries):
    Standard ID agrees: 0/3 (0%)
    Queries: [3, 4, 5]

Prediction accuracy:
  Cyclic ID matched expectations: 6/6 (100%)

Performance:
  Avg Cyclic ID time: 0.0037s
  Avg Standard ID time: 0.0007s

COMPLETE RESULTS TABLE

 Q#                   Query                                         Description       Expected      Cyclic ID  Standard ID  Agreement
  1       P(soxS | do(fnr))  Single intervention outside cycle → target outside   IDENTIFIABLE   IDENTIFIABLE IDENTIFIABLE       True
  2       P(oxyR | do(fnr)) Single intervention outside cycle → target in cycle   IDENTIFIABLE   IDENTIFIABLE IDENTIFIABLE       True
  3       P(oxyR | do(fur))      Single intervention in cycle → target in cycle UNIDENTIFIA

In [ ]:
# Cyclic ID Algorithm debugging trace with example query

print("="*70)
print("ALGORITHM Trace P(oxyR | do(fur))")
print("="*70)

W = {Variable('fur')}
Y = {Variable('oxyR')}

print(f"\nQuery: P(oxyR | do(fur))")
print(f"W = {{{', '.join([str(v) for v in W])}}}")
print(f"Y = {{{', '.join([str(v) for v in Y])}}}")

# Line 2: Precondition Check
print(f"\n[Line 2] Preconditions: Y⊆V, W⊆V, Y∩W=∅")
print(f"  Expected: True, True, True")
print(f"  Actual: {Y.issubset(set(stress_graph.nodes()))}, {W.issubset(set(stress_graph.nodes()))}, {not (Y & W)}")
print(f"  Status: All satisfied")

# Line 3: Ancestral closure H
print(f"\n[Line 3] Ancestral closure H in G\\W")
G_minus_W = stress_graph.remove_nodes_from(W)
H = G_minus_W.ancestors_inclusive(Y)
H_genes = sorted([str(v) for v in H])

print(f"  Expected: H includes ancestors of oxyR, fur not in H")
print(f"  Actual: H = {{{', '.join(H_genes)}}} (|H| = {len(H)})")
print(f"  Actual: fur in H = {Variable('fur') in H}")

# Line 4: Consolidated Districts
print(f"\n[Line 4] Consolidated districts of H")
H_subgraph = G_minus_W.subgraph(H)
districts = get_graph_consolidated_districts(H_subgraph)
district_sizes = sorted([len(d) for d in districts], reverse=True)

print(f"  Expected: H decomposes into districts")
print(f"  Actual: {len(districts)} districts, sizes: {district_sizes}")

for i, district_c in enumerate(districts, 1):
    genes = sorted([str(v) for v in district_c])
    print(f"    District {i}: {{{', '.join(genes)}}}")

# Lines 5-20: District loop to use IDCD to check each district
print(f"\n[Lines 5-20] Check each district for A=D condition")

for i, district_c in enumerate(districts, 1):
    C_genes = sorted([str(v) for v in district_c])
    print(f"\n  District {i}: C = {{{', '.join(C_genes)}}}")
    
    # Get consolidated district D in full graph
    D = get_consolidated_district(stress_graph, district_c)
    D_genes = sorted([str(v) for v in D])
    print(f"    Expected: D is consolidated district of C in full graph G")
    print(f"    Actual: D = {{{', '.join(D_genes)}}} (|D| = {len(D)})")
    
    # Compute ancestral closure A within district
    D_subgraph = stress_graph.subgraph(D)
    A = D_subgraph.ancestors_inclusive(district_c) & D
    A_genes = sorted([str(v) for v in A])
    print(f"\n    Expected: A = An^G[D](C) ∩ D")
    print(f"    Actual: A = {{{', '.join(A_genes)}}} (|A| = {len(A)})")
    
    # CHECK: A = D?
    print(f"\n    Expected: If A = D, then Line 19-20 triggers unidentifiable")
    print(f"    Actual: A = D? {A == D}")
    print(f"    Actual: |A| = {len(A)}, |D| = {len(D)}")
    
    if A == D:
        print(f"\n    Status: A = D condition triggered (Line 19-20)")
        print(f"    Reason: All nodes in D are mutual ancestors")
        print(f"    Mechanism: 3-gene feedback loop (fur <-> soxR <-> oxyR)")
        print(f"    Result: unidentifiable")
    else:
        print(f"    Status: A is proper subset of D, algorithm continues")





ALGORITHM TRACE: P(oxyR | do(fur))

Query: P(oxyR | do(fur))
W = {fur}
Y = {oxyR}

[Line 2] Preconditions: Y⊆V, W⊆V, Y∩W=∅
  Expected: True, True, True
  Actual: True, True, True
  Status: All satisfied

[Line 3] Ancestral closure H in G\W
  Expected: H includes ancestors of oxyR, fur not in H
  Actual: H = {fnr, oxyR, soxR} (|H| = 3)
  Actual: fur in H = False

[Line 4] Consolidated districts of H
  Expected: H decomposes into districts
  Actual: 3 districts, sizes: [1, 1, 1]
    District 1: {soxR}
    District 2: {fnr}
    District 3: {oxyR}

[Lines 5-20] Check each district for A=D condition

  District 1: C = {soxR}
    Expected: D is consolidated district of C in full graph G
    Actual: D = {fur, oxyR, soxR} (|D| = 3)

    Expected: A = An^G[D](C) ∩ D
    Actual: A = {fur, oxyR, soxR} (|A| = 3)

    Expected: If A = D, then Line 19-20 triggers unidentifiable
    Actual: A = D? True
    Actual: |A| = 3, |D| = 3

    Status: A = D condition triggered (Line 19-20)
    Reason: All no